# Todo App với Streamlit + FastAPI + Firebase

**Môn học:** Hệ thống phân tán / Lập trình ứng dụng web  
**Họ và tên:** Lê Văn Quốc  
**MSSV:** 24120421  
**Trường:** Đại học Khoa học Tự nhiên TP.HCM (HCMUS)

---
## 1. Mô tả mục tiêu

Bài tập yêu cầu xây dựng một ứng dụng **Todo App** hoàn chỉnh với kiến trúc hiện đại, gồm:

- **Frontend:** Streamlit — giao diện người dùng đơn giản, trực quan
- **Backend:** FastAPI (ASGI) + Uvicorn — xử lý logic, xác thực, cung cấp REST API
- **Dịch vụ Firebase:** Authentication (xác thực người dùng) + Cloud Firestore (lưu trữ dữ liệu)

### Tính năng bắt buộc
1. Đăng ký và đăng nhập bằng email/password (Firebase Authentication)
2. Thêm, xem, cập nhật (hoàn thành), xóa công việc Todo (CRUD)
3. Chỉ hiển thị danh sách công việc của người dùng đang đăng nhập

### Luồng dữ liệu tổng quát
```
User → Streamlit → FastAPI (verify Firebase ID Token) → Firestore
```

---
## 2. Kiến trúc tổng quát

```
┌─────────────────────────────────────────────────────────┐
│                      CLIENT BROWSER                     │
│                   localhost:8501                         │
└──────────────────────┬──────────────────────────────────┘
                       │ HTTP requests
┌──────────────────────▼──────────────────────────────────┐
│               FRONTEND: Streamlit                        │
│  - Giao diện đăng nhập / đăng ký                        │
│  - Hiển thị danh sách Todo                              │
│  - Gửi Firebase ID Token trong header Authorization     │
└──────────────────────┬──────────────────────────────────┘
                       │ REST API (Bearer Token)
┌──────────────────────▼──────────────────────────────────┐
│               BACKEND: FastAPI + Uvicorn                 │
│  - POST /auth/register  → Tạo user Firebase Auth        │
│  - POST /auth/login     → Trả về ID Token               │
│  - GET  /todos          → Lấy danh sách todo của user   │
│  - POST /todos          → Tạo todo mới                  │
│  - PUT  /todos/{id}     → Cập nhật todo                 │
│  - DELETE /todos/{id}   → Xóa todo                      │
└──────────┬────────────────────────┬────────────────────┘
           │ verify_id_token        │ read/write
┌──────────▼──────────┐  ┌─────────▼──────────────────────┐
│  Firebase Auth      │  │  Cloud Firestore               │
│  Emulator :9099     │  │  Emulator :8080                │
│  - Quản lý user     │  │  Collection: todos             │
│  - Cấp ID Token     │  │  Fields: title, done, uid      │
└─────────────────────┘  └────────────────────────────────┘
```

### Mô hình dữ liệu Firestore

Collection: `todos`  
Document structure:
```json
{
  "uid": "DS71Z3cNvbjzCU1R9HnXZdMRcFdQ",
  "title": "Học FastAPI",
  "done": false
}
```

---
## 3. Thiết lập Firebase

### 3.1 Tạo Firebase Project
1. Truy cập https://console.firebase.google.com
2. Tạo project mới: `todo-streamlit-fastapi`
3. Bật **Authentication** → Sign-in method → **Email/Password**
4. Tạo **Cloud Firestore** database (Standard edition, asia-southeast1)

### 3.2 Sử dụng Firebase Emulator (thay cho Firestore cloud)

Do giới hạn billing của Firebase Spark plan, project sử dụng **Firebase Emulator Suite** để chạy Auth và Firestore locally.

**Cài đặt:**
```bash
npm install -g firebase-tools
firebase login
firebase init emulators  # chọn Authentication + Firestore
firebase emulators:start
```

**Emulator endpoints:**
- Authentication: `http://127.0.0.1:9099`
- Firestore: `http://127.0.0.1:8080`
- Emulator UI: `http://127.0.0.1:4000`

### 3.3 Cấu hình backend kết nối emulator
```python
import os
os.environ["FIREBASE_AUTH_EMULATOR_HOST"] = "127.0.0.1:9099"
os.environ["FIRESTORE_EMULATOR_HOST"] = "127.0.0.1:8080"
```

---
## 4. Thiết kế Backend (FastAPI)

### 4.1 Cấu trúc thư mục
```
backend/
├── venv/
├── main.py
└── requirements.txt
```

### 4.2 Thư viện sử dụng
```
fastapi
uvicorn
firebase-admin
google-cloud-firestore
python-jose[cryptography]
requests
```

### 4.3 Code Backend (main.py)

In [17]:
# backend/main.py
# Đây là code minh họa — chạy thực tế từ terminal

backend_code = '''
from fastapi import FastAPI, HTTPException, Depends, Header
from fastapi.middleware.cors import CORSMiddleware
import firebase_admin
from firebase_admin import auth, credentials
from google.cloud import firestore
from pydantic import BaseModel
import os

# Kết nối Firebase Emulator
os.environ["FIREBASE_AUTH_EMULATOR_HOST"] = "127.0.0.1:9099"
os.environ["FIRESTORE_EMULATOR_HOST"] = "127.0.0.1:8080"

cred = credentials.ApplicationDefault()
firebase_admin.initialize_app(cred, {"projectId": "todo-streamlit-fastapi"})
db = firestore.Client(project="todo-streamlit-fastapi")

app = FastAPI()

# Cho phép CORS từ Streamlit
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

# Dependency: Xác thực Firebase ID Token
async def get_current_user(authorization: str = Header(...)):
    try:
        token = authorization.replace("Bearer ", "")
        decoded = auth.verify_id_token(token)
        return decoded["uid"]
    except Exception:
        raise HTTPException(status_code=401, detail="Invalid token")

class AuthRequest(BaseModel):
    email: str
    password: str

class TodoRequest(BaseModel):
    title: str
    done: bool = False

# Đăng ký user mới
@app.post("/auth/register")
async def register(body: AuthRequest):
    try:
        user = auth.create_user(email=body.email, password=body.password)
        return {"uid": user.uid, "email": user.email}
    except Exception as e:
        raise HTTPException(status_code=400, detail=str(e))

# Đăng nhập - trả về Firebase ID Token
@app.post("/auth/login")
async def login(body: AuthRequest):
    import requests as req
    url = "http://127.0.0.1:9099/identitytoolkit.googleapis.com/v1/accounts:signInWithPassword?key=fake-api-key"
    res = req.post(url, json={"email": body.email, "password": body.password, "returnSecureToken": True})
    data = res.json()
    if "error" in data:
        raise HTTPException(status_code=400, detail=data["error"]["message"])
    return {"idToken": data["idToken"], "uid": data["localId"]}

# Lấy danh sách todo của user đang đăng nhập
@app.get("/todos")
async def get_todos(uid: str = Depends(get_current_user)):
    todos_ref = db.collection("todos").where("uid", "==", uid).stream()
    return [{"id": doc.id, **doc.to_dict()} for doc in todos_ref]

# Tạo todo mới
@app.post("/todos")
async def create_todo(body: TodoRequest, uid: str = Depends(get_current_user)):
    doc_ref = db.collection("todos").document()
    doc_ref.set({"title": body.title, "done": body.done, "uid": uid})
    return {"id": doc_ref.id, "title": body.title, "done": body.done}

# Cập nhật todo (đánh dấu hoàn thành)
@app.put("/todos/{todo_id}")
async def update_todo(todo_id: str, body: TodoRequest, uid: str = Depends(get_current_user)):
    doc_ref = db.collection("todos").document(todo_id)
    doc_ref.update({"title": body.title, "done": body.done})
    return {"id": todo_id, "title": body.title, "done": body.done}

# Xóa todo
@app.delete("/todos/{todo_id}")
async def delete_todo(todo_id: str, uid: str = Depends(get_current_user)):
    db.collection("todos").document(todo_id).delete()
    return {"message": "Deleted"}
'''

print("Backend code loaded. Chạy bằng lệnh:")
print("  cd backend")
print("  uvicorn main:app --reload --port 8000")

Backend code loaded. Chạy bằng lệnh:
  cd backend
  uvicorn main:app --reload --port 8000


### 4.4 Giải thích luồng xác thực

1. **Frontend** gọi `/auth/login` với email/password
2. **Backend** gọi Firebase Auth Emulator → nhận **ID Token** (JWT)
3. **Frontend** lưu ID Token vào `st.session_state`
4. Mỗi request tiếp theo, Frontend gửi token trong header: `Authorization: Bearer <idToken>`
5. **Backend** dùng `firebase_admin.auth.verify_id_token()` để xác thực token → lấy `uid`
6. Dùng `uid` để query/write Firestore đúng data của user

### 4.5 Swagger UI

FastAPI tự động sinh tài liệu API tại `http://127.0.0.1:8000/docs`:

| Method | Endpoint | Mô tả |
|--------|----------|-------|
| POST | /auth/register | Đăng ký user mới |
| POST | /auth/login | Đăng nhập, trả về ID Token |
| GET | /todos | Lấy danh sách todo (cần token) |
| POST | /todos | Tạo todo mới (cần token) |
| PUT | /todos/{id} | Cập nhật todo (cần token) |
| DELETE | /todos/{id} | Xóa todo (cần token) |

---
## 5. Thiết kế Frontend (Streamlit)

### 5.1 Cấu trúc thư mục
```
frontend/
├── venv/
└── app.py
```

### 5.2 Thư viện sử dụng
```
streamlit
requests
```

### 5.3 Code Frontend (app.py)

In [18]:
# frontend/app.py
# Đây là code minh họa — chạy thực tế từ terminal

frontend_code = '''
import streamlit as st
import requests

API = "http://127.0.0.1:8000"

st.title("📝 Todo App")

# Khởi tạo session state
if "token" not in st.session_state:
    st.session_state.token = None
if "uid" not in st.session_state:
    st.session_state.uid = None

# === CHƯA ĐĂNG NHẬP ===
if not st.session_state.token:
    tab1, tab2 = st.tabs(["Đăng nhập", "Đăng ký"])

    with tab1:
        email = st.text_input("Email", key="login_email")
        password = st.text_input("Mật khẩu", type="password", key="login_pass")
        if st.button("Đăng nhập"):
            res = requests.post(f"{API}/auth/login",
                                json={"email": email, "password": password})
            if res.status_code == 200:
                data = res.json()
                st.session_state.token = data["idToken"]
                st.session_state.uid = data["uid"]
                st.rerun()
            else:
                st.error(res.json().get("detail", "Lỗi đăng nhập"))

    with tab2:
        email = st.text_input("Email", key="reg_email")
        password = st.text_input("Mật khẩu", type="password", key="reg_pass")
        if st.button("Đăng ký"):
            res = requests.post(f"{API}/auth/register",
                                json={"email": email, "password": password})
            if res.status_code == 200:
                st.success("Đăng ký thành công! Hãy đăng nhập.")
            else:
                st.error(res.json().get("detail", "Lỗi đăng ký"))

# === ĐÃ ĐĂNG NHẬP ===
else:
    st.success(f"Đã đăng nhập: {st.session_state.uid}")
    if st.button("Đăng xuất"):
        st.session_state.token = None
        st.session_state.uid = None
        st.rerun()

    headers = {"Authorization": f"Bearer {st.session_state.token}"}

    # Thêm todo
    st.subheader("➕ Thêm công việc")
    new_title = st.text_input("Tên công việc")
    if st.button("Thêm"):
        res = requests.post(f"{API}/todos",
                            json={"title": new_title, "done": False},
                            headers=headers)
        if res.status_code == 200:
            st.success("Đã thêm!")
            st.rerun()

    # Danh sách todo
    st.subheader("📋 Danh sách công việc")
    res = requests.get(f"{API}/todos", headers=headers)
    if res.status_code == 200:
        todos = res.json()
        for todo in todos:
            col1, col2, col3 = st.columns([5, 2, 1])
            with col1:
                st.write(f"{\"✅\" if todo[\'done\'] else \'⬜\'} {todo[\'title\']}")
            with col2:
                label = "Bỏ hoàn thành" if todo["done"] else "Hoàn thành"
                if st.button(label, key=f"done_{todo[\'id\']}"):
                    requests.put(f"{API}/todos/{todo[\'id\']}",
                                 json={"title": todo["title"], "done": not todo["done"]},
                                 headers=headers)
                    st.rerun()
            with col3:
                if st.button("🗑️", key=f"del_{todo[\'id\']}"):
                    requests.delete(f"{API}/todos/{todo[\'id\']}", headers=headers)
                    st.rerun()
'''

print("Frontend code loaded. Chạy bằng lệnh:")
print("  cd frontend")
print("  streamlit run app.py")

Frontend code loaded. Chạy bằng lệnh:
  cd frontend
  streamlit run app.py


### 5.4 Giải thích luồng Frontend

- **`st.session_state`**: Lưu trữ `token` và `uid` giữa các lần render
- **Tab Đăng nhập / Đăng ký**: Tách biệt 2 chức năng auth
- **`st.rerun()`**: Refresh giao diện sau mỗi thao tác
- **`headers`**: Gửi Bearer Token trong mọi request đến backend
- **`col1, col2, col3`**: Layout 3 cột cho mỗi todo item

---
## 6. Hướng dẫn chạy ứng dụng

### Bước 1: Khởi động Firebase Emulator
```bash
cd todo-streamlit-fastapi-firebase
firebase emulators:start
```
→ Emulator UI: http://127.0.0.1:4000

### Bước 2: Khởi động Backend
```bash
cd backend
venv\Scripts\activate       # Windows
uvicorn main:app --reload --port 8000
```
→ API Docs: http://127.0.0.1:8000/docs

### Bước 3: Khởi động Frontend
```bash
cd frontend
venv\Scripts\activate       # Windows
streamlit run app.py
```
→ App: http://localhost:8501

### Cấu trúc thư mục cuối cùng
```
todo-streamlit-fastapi-firebase/
├── backend/
│   ├── venv/
│   └── main.py
├── frontend/
│   ├── venv/
│   └── app.py
├── notebook/
│   └── todo_app_notebook.ipynb  ← file này
├── firebase.json
└── .firebaserc
```

---
## 7. Kiểm thử cơ bản

### Test 1: Đăng ký tài khoản

In [19]:
import requests

API = "http://127.0.0.1:8000"

# Test đăng ký
res = requests.post(f"{API}/auth/register", json={
    "email": "test@example.com",
    "password": "123456"
})
print("Đăng ký:", res.status_code, res.json())

Đăng ký: 400 {'detail': 'The user with the provided email already exists (EMAIL_EXISTS).'}


**Kết quả mong đợi:**
```
Đăng ký: 200 {'uid': '...', 'email': 'test@example.com'}
```

In [20]:
# Test đăng nhập
res = requests.post(f"{API}/auth/login", json={
    "email": "test@example.com",
    "password": "123456"
})
data = res.json()
print("Đăng nhập:", res.status_code)
print("UID:", data.get("uid"))
print("Token (50 ký tự đầu):", data.get("idToken", "")[:50] + "...")

TOKEN = data.get("idToken")
headers = {"Authorization": f"Bearer {TOKEN}"}

Đăng nhập: 200
UID: jg3GHPvtvpoRxp0XgazSZnxSJPB3
Token (50 ký tự đầu): eyJhbGciOiJub25lIiwidHlwIjoiSldUIn0.eyJlbWFpbCI6In...


**Kết quả mong đợi:**
```
Đăng nhập: 200
UID: DS71Z3cNvbjzCU1R9HnXZdMRcFdQ
Token (50 ký tự đầu): eyJhbGciOiJSUzI1NiIsImtpZCI6Ij...
```

In [21]:
# Test tạo todo
res = requests.post(f"{API}/todos",
    json={"title": "Học FastAPI", "done": False},
    headers=headers)
print("Tạo Todo:", res.status_code, res.json())

Tạo Todo: 200 {'id': 'GqU4lW2oMx4X4w4sHWyZ', 'title': 'Học FastAPI', 'done': False}


**Kết quả mong đợi:**
```
Tạo Todo: 200 {'id': 'abc123...', 'title': 'Học FastAPI', 'done': False}
```

In [22]:
# Test lấy danh sách todo
res = requests.get(f"{API}/todos", headers=headers)
todos = res.json()
print("Danh sách Todo:", res.status_code)
for todo in todos:
    status = "✅" if todo["done"] else "⬜"
    print(f"  {status} [{todo['id'][:8]}...] {todo['title']}")

Danh sách Todo: 200
  ⬜ [GqU4lW2o...] Học FastAPI


In [23]:
# Test cập nhật todo (đánh dấu hoàn thành)
if todos:
    todo_id = todos[0]["id"]
    res = requests.put(f"{API}/todos/{todo_id}",
        json={"title": todos[0]["title"], "done": True},
        headers=headers)
    print("Cập nhật Todo:", res.status_code, res.json())

Cập nhật Todo: 200 {'id': 'GqU4lW2oMx4X4w4sHWyZ', 'title': 'Học FastAPI', 'done': True}


In [24]:
# Test xóa todo
if todos:
    todo_id = todos[0]["id"]
    res = requests.delete(f"{API}/todos/{todo_id}", headers=headers)
    print("Xóa Todo:", res.status_code, res.json())

# Kiểm tra danh sách sau khi xóa
res = requests.get(f"{API}/todos", headers=headers)
print("Danh sách sau khi xóa:", res.json())

Xóa Todo: 200 {'message': 'Deleted'}
Danh sách sau khi xóa: []


### Tổng kết kết quả kiểm thử

| Chức năng | Endpoint | Kết quả |
|-----------|----------|---------|
| Đăng ký | POST /auth/register | ✅ 200 OK |
| Đăng nhập | POST /auth/login | ✅ 200 OK + ID Token |
| Tạo Todo | POST /todos | ✅ 200 OK |
| Xem Todo | GET /todos | ✅ 200 OK |
| Cập nhật Todo | PUT /todos/{id} | ✅ 200 OK |
| Xóa Todo | DELETE /todos/{id} | ✅ 200 OK |
| Bảo mật (không có token) | GET /todos | ✅ 401 Unauthorized |

---
## 8. Kết luận

Dự án đã xây dựng thành công một **Todo App** hoàn chỉnh với kiến trúc hiện đại:

### Những gì đã đạt được
- ✅ **Firebase Authentication** với Firebase Emulator — xác thực người dùng bằng email/password
- ✅ **Cloud Firestore** (Emulator) — lưu trữ dữ liệu theo mô hình collection-document
- ✅ **FastAPI** — REST API với ASGI/Uvicorn, xác thực JWT token, tài liệu Swagger tự động
- ✅ **Streamlit** — giao diện người dùng responsive, quản lý state bằng session_state
- ✅ **Phân quyền dữ liệu** — mỗi user chỉ thấy Todo của chính mình

### Hướng mở rộng
- Thêm chức năng lọc/tìm kiếm theo tiêu đề
- Tích hợp đăng nhập bằng Google
- Deploy lên cloud (Firebase Hosting + Cloud Run)
- Thêm deadline và priority cho mỗi Todo